##  20 TOC check — look only, saves nothing

In [11]:
# 20.0 TOC check — look only, saves nothing
import re, json
import pandas as pd
from IPython.display import display
from src.paths import TOC_CSV, PAGES_FINAL

pages  = [json.loads(line) for line in PAGES_FINAL.open(encoding="utf-8")]
PAGE   = {p["pdf_page"]: p for p in pages}
SEARCH = sorted(p["pdf_page"] for p in pages if p["search"])

toc = pd.read_csv(TOC_CSV, dtype=str)
for col in ["level", "printed_start", "printed_end", "pdf_start"]:
    toc[col] = pd.to_numeric(toc[col], errors="coerce")
toc = toc[toc["kind"] != "letter"].reset_index(drop=True)        # letters are done → letters.jsonl
toc["pdf_end"] = toc["printed_end"] + 20
print(toc["kind"].value_counts().to_string(), "\n")

# ---------- A. numbers make sense ----------
def parent_key(r):
    return f"{'Chapter' if r['kind'] == 'chapter' else 'Annexure'} {r['number']}"
TOP = {parent_key(r): r for _, r in toc[toc["level"] == 0].iterrows()}

problems = []
for i, r in toc.iterrows():
    if r["pdf_start"] != r["printed_start"] + 20:
        problems.append((r["number"], r["title"], "pdf_start != printed_start + 20"))
    if r["printed_end"] < r["printed_start"]:
        problems.append((r["number"], r["title"], "ends before it starts"))
    if i and r["pdf_start"] < toc.loc[i - 1, "pdf_start"]:
        problems.append((r["number"], r["title"], "starts before the row above it"))
    if r["level"] > 0:
        par = TOP.get(r["parent"])
        if par is None:
            problems.append((r["number"], r["title"], f"parent {r['parent']!r} not found"))
        elif not (par["printed_start"] <= r["printed_start"] <= r["printed_end"] <= par["printed_end"]):
            problems.append((r["number"], r["title"], f"outside parent {r['parent']}"))
print(f"A. number problems: {len(problems)}")
if problems:
    display(pd.DataFrame(problems, columns=["number", "title", "problem"]))

# ---------- B. every searchable page inside a top-level range ----------
covered = {n for _, r in toc[toc["level"] == 0].iterrows()
           for n in range(int(r["pdf_start"]), int(r["pdf_end"]) + 1)}
outside = [n for n in SEARCH if n not in covered]
print(f"\nB. searchable pages outside every chapter/annexure range: {outside}")

# ---------- C. TOC entry found as a heading line? ----------
HEADING = {"L0", "L1", "L2", "L3"}
norm = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def headings(pdf_no):
    p = PAGE.get(pdf_no)
    return [] if p is None else [l["text"] for l in p["lines"] if l["level"] in HEADING]

def find_heading(r):
    num, title = ("" if pd.isna(r["number"]) else str(r["number"])), norm(r["title"])
    for offset in (0, 1, -1):                       # its own page first, then the neighbours
        for h in headings(int(r["pdf_start"]) + offset):
            nh = norm(h)
            by_num   = r["kind"] == "section" and bool(re.match(rf"^{re.escape(num)}\.?(\s|$)", h.strip()))
            by_title = len(nh) >= 8 and (title in nh or nh in title)
            if by_num or by_title:
                return offset, h
    return None, None

found = toc.apply(lambda r: pd.Series(find_heading(r), index=["offset", "heading"]), axis=1)
check = pd.concat([toc[["kind", "number", "title", "pdf_start"]], found], axis=1)
print(f"\nC. heading found: {check['heading'].notna().sum()}/{len(check)}  "
      f"(own page: {(check['offset'] == 0).sum()}, next/previous page: {check['offset'].isin([1, -1]).sum()})")
missing = check[check["heading"].isna()].copy()
missing["headings_on_page"] = missing["pdf_start"].map(lambda n: headings(int(n))[:3])
if len(missing):
    display(missing[["kind", "number", "title", "pdf_start", "headings_on_page"]])

# ---------- D. numbered headings that the TOC does not list ----------
TOC_NUMS = set(toc["number"].dropna().astype(str))
extra = []
for n in SEARCH:
    for l in PAGE[n]["lines"]:
        if l["level"] in HEADING:
            m = re.match(r"^(\d+(?:\.\d+)*)\.?\s", l["text"].strip())
            if m and m[1] not in TOC_NUMS:
                extra.append((n, l["level"], l["text"][:80]))
print(f"\nD. numbered headings not in the TOC: {len(extra)}")
if extra:
    display(pd.DataFrame(extra, columns=["pdf_page", "level", "heading"]).head(40))

kind
section            59
annexure item      28
chapter             9
case study part     7
annexure            3 

A. number problems: 0

B. searchable pages outside every chapter/annexure range: [15, 16]

C. heading found: 104/106  (own page: 104, next/previous page: 0)


,kind,number,title,pdf_start,headings_on_page
88,annexure,II,Compendium of Letters,109,[]
92,case study part,2,Swasthya Sewa Dapoon: An Integrated MIS-GIS Sy...,148,"[Advantages and Scalability, Part 2: SwasthyaS..."



D. numbered headings not in the TOC: 0


##  20.0 step-up page

In [12]:
import re, json
from collections import Counter
import pandas as pd
from IPython.display import display
from src.paths import TOC_CSV, PAGES_FINAL, SECTIONS

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]

PAGES   = read_jsonl(PAGES_FINAL)
PAGE    = {p["pdf_page"]: p for p in PAGES}
SEARCH  = sorted(p["pdf_page"] for p in PAGES if p["search"])
HEADING = {"L0", "L1", "L2", "L3"}

toc = pd.read_csv(TOC_CSV, dtype=str)
for col in ["level", "printed_start", "printed_end", "pdf_start"]:
    toc[col] = pd.to_numeric(toc[col], errors="coerce")
SKIP    = (toc["kind"] == "letter") | ((toc["kind"] == "annexure") & (toc["number"] == "II"))   # → letters.jsonl
SEC_TOC = toc[~SKIP].reset_index(drop=True)

def clean_row(r):
    """pandas row → plain dict (plain int / str / None, so it saves to JSON)."""
    return {"kind": r["kind"], "number": None if pd.isna(r["number"]) else str(r["number"]),
            "title": r["title"], "level": int(r["level"]),
            "printed_start": int(r["printed_start"]), "printed_end": int(r["printed_end"]),
            "pdf_start": int(r["pdf_start"])}

TOC_ROWS = [clean_row(r) for _, r in SEC_TOC.iterrows()]

# front matter the TOC does not cover (TOC check B: PDF 15-16). Must come first: it is the earliest page.
MANUAL_SECTIONS = [
    {"kind": "glossary", "number": None, "title": "List of Abbreviations", "level": 0,
     "printed_start": None, "printed_end": None, "pdf_start": 15, "line": 0},
]
print(f"{len(SEARCH)} searchable pages · {len(TOC_ROWS)} TOC rows + {len(MANUAL_SECTIONS)} manual")

114 searchable pages · 105 TOC rows + 1 manual


## 20.1  Names and heading matching

In [13]:
norm       = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()
squash     = lambda s: norm(s).replace(" ", "")                 # "Swasthya Sewa" == "SwasthyaSewa"
# one piece of a label above a chapter/annexure title: "Annexure I" · "Annexure" · "I" / "III" · "Chapter 5" · "5"
LABEL_PART = re.compile(r"^((chapter|annexure)(\s*[-:]?\s*[0-9ivx]+)?|[0-9]+|[ivx]+)\s*$", re.I)

LABEL = {   # registry: kind → how the section is named in its path (unknown kind → KeyError)
    "chapter":         lambda r: f"Chapter {r['number']}: {r['title']}",
    "section":         lambda r: f"{r['number']} {r['title']}",
    "annexure":        lambda r: f"Annexure {r['number']}: {r['title']}",
    "annexure item":   lambda r: r["title"],
    "case study part": lambda r: f"Part {r['number']}: {r['title']}",
    "glossary":        lambda r: r["title"],
}

def match_score(row, text):
    """How well one heading line matches a TOC row:
    3 = same title · 2 = same number / 'Part N' · 1 = title inside the line · 0 = no match."""
    t, title, num = squash(text), squash(row["title"]), row["number"] or ""
    if t == title:
        return 3
    if row["kind"] == "section" and num and re.match(rf"^{re.escape(num)}\.?(\s|$)", text.strip()):
        return 2
    if row["kind"] == "case study part" and num and re.match(rf"^part\s*{re.escape(num)}\b", text.strip(), flags=re.I):
        return 2
    if len(t) >= 8 and title and title in t:
        return 1
    return 0

## 20.2 Find where each section starts


In [14]:
def find_start(row, after):
    """(pdf_page, line_index) of the row's heading on its start page, AFTER the previous start.
    Tuples compare item by item: (32, 5) > (32, 2) and (33, 0) > (32, 40)."""
    page, lines = row["pdf_start"], PAGE[row["pdf_start"]]["lines"]
    scored = [(match_score(row, l["text"]), i) for i, l in enumerate(lines)
              if l["level"] in HEADING and (page, i) > after]
    scored = [(s, i) for s, i in scored if s > 0]
    if not scored:
        return None
    best = max(s for s, _ in scored)
    i = min(i for s, i in scored if s == best)                  # best score, earliest line
    if row["kind"] in ("chapter", "annexure"):
        # take the label above the title too: "Annexure I" on one line, or "Annexure" + "I" on two lines
        while i > 0 and (page, i - 1) > after and LABEL_PART.match(lines[i - 1]["text"].strip()):
            i -= 1
    return page, i

STARTS, NOT_FOUND, after = [], [], (0, -1)
for row in MANUAL_SECTIONS + TOC_ROWS:                           # TOC order = page order
    pos = (row["pdf_start"], row["line"]) if "line" in row else find_start(row, after)
    if pos is None:
        NOT_FOUND.append((row["kind"], row["number"], row["title"], row["pdf_start"]))
        continue
    STARTS.append({**{k: v for k, v in row.items() if k != "line"}, "pdf_page": pos[0], "line": pos[1]})
    after = pos

# ids, names and path (a stack: pop everything at the same or deeper level, then push)
stack = []
for k, s in enumerate(STARTS):
    s["section_id"] = f"S{k + 1:03d}"
    s["label"] = LABEL[s["kind"]](s)
    while stack and stack[-1]["level"] >= s["level"]:
        stack.pop()
    s["parent_id"] = stack[-1]["section_id"] if stack else None
    s["path"] = [x["label"] for x in stack] + [s["label"]]
    stack.append(s)

print(f"{len(STARTS)} section starts · not found: {NOT_FOUND}")

106 section starts · not found: []


## 20.3 Walk every line → sections

In [15]:
TABLES     = {t["id"]: t for p in PAGES for t in p["tables"]}
TABLE_LINE = re.compile(r"^\[table ([^\]]+)\]$")                # "[table 86-1]" → "86-1"
START_AT   = {(s["pdf_page"], s["line"]): k for k, s in enumerate(STARTS)}
KEEP       = ["section_id", "kind", "number", "title", "level", "label", "path", "parent_id",
              "printed_start", "printed_end"]

def build_sections():
    secs = [{**{k: s[k] for k in KEEP}, "lines": []} for s in STARTS]
    current, unassigned = None, []
    for n in SEARCH:
        for i, l in enumerate(PAGE[n]["lines"]):
            current = START_AT.get((n, i), current)              # a start here? switch section, else keep
            if current is None:
                unassigned.append((n, i, l["text"][:60]))
                continue
            secs[current]["lines"].append({"pdf_page": n, "level": l["level"], "text": l["text"], "top": l.get("top")})
    for s in secs:
        pages = sorted({l["pdf_page"] for l in s["lines"]})
        table_ids = [TABLE_LINE.match(l["text"])[1] for l in s["lines"] if l["level"] == "table"]
        s["pdf_pages"]     = pages
        s["printed_pages"] = [PAGE[n]["printed_page"] for n in pages]
        s["tables"]        = [TABLES[t] for t in table_ids]
        # footnote markers were removed in step 13, so a page's footnotes go to every section on that page
        s["footnotes"]     = [{"pdf_page": n, **f} for n in pages for f in PAGE[n]["footnotes"]]
        s["n_chars"]       = (sum(len(l["text"]) for l in s["lines"])
                              + sum(len(" ".join(t.get("texts", []))) for t in s["tables"]))
    return secs, unassigned

SECTIONS_OUT, UNASSIGNED = build_sections()
print(f"{len(SECTIONS_OUT)} sections · {len(UNASSIGNED)} lines without a section")

106 sections · 0 lines without a section


## 20.4 Look (changes nothing)

In [16]:
overview = pd.DataFrame([{"id": s["section_id"], "kind": s["kind"], "label": s["label"][:55],
                          "pdf": f"{s['pdf_pages'][0]}-{s['pdf_pages'][-1]}" if s["pdf_pages"] else "-",
                          "lines": len(s["lines"]), "tables": len(s["tables"]), "chars": s["n_chars"]}
                         for s in SECTIONS_OUT])
print(overview["kind"].value_counts().to_string())
print("\nchars per section:", overview["chars"].describe().round(0).to_dict())
display(overview.sort_values("chars", ascending=False).head(10))     # the big ones → split in step 21

def show_section(key, k=12):
    s = next(x for x in SECTIONS_OUT if key in (x["number"], x["title"], x["section_id"]))
    print(" > ".join(s["path"]), f"| PDF {s['pdf_pages']} | printed {s['printed_pages']}")
    for l in s["lines"][:k]:
        print(f"  p{l['pdf_page']} {l['level']:<8} {l['text'][:90]}")

show_section("5.3")

kind
section            59
annexure item      28
chapter             9
case study part     7
annexure            2
glossary            1

chars per section: {'count': 106.0, 'mean': 2196.0, 'std': 4333.0, 'min': 29.0, '25%': 587.0, '50%': 1047.0, '75%': 2242.0, 'max': 40259.0}


,id,kind,label,pdf,lines,tables,chars
72,S073,annexure item,Sample ToRs/ Roles and Responsibilities,54-70,656,0,40259
78,S079,annexure item,Process of Performance Appraisal,80-87,194,11,14203
86,S087,annexure item,Process of Rationalization,101-102,49,2,10593
75,S076,annexure item,Contents of the Contract,74-77,156,1,10228
88,S089,annexure item,Exit Policy,104-108,72,8,5779
97,S098,annexure item,Appraisals and Rationalization of Human Resour...,155-157,70,1,5085
98,S099,annexure item,Person-Centred Care through Shared leadership-...,158-160,78,0,4904
73,S074,annexure item,Stages of Selection,71-72,74,0,4891
103,S104,annexure item,AMANAT - Nurse Mobile Mentoring Programme - Bihar,164-165,64,0,4848
71,S072,annexure item,Process of Designing Job Description/ToR,52-53,24,1,4646


Chapter 5: Recruitment > 5.3 Process of Recruitment | PDF [32] | printed ['12']
  p32 L1       5.3. Process of Recruitment
  p32 body     The recruitment process should be relevant, aptitude/ skill-based, transparentand equitabl
  p32 body     should be applied consistently to all candidates.
  p32 body     The key steps in the process of recruitment are shown in Exhibit 1 and explained below:
  p32 caption  Exhibit 1: Key steps in the Process of Recruitment
  p32 diagram  1. Identify and validate vacancies.
  p32 diagram  2. Review Job Description.
  p32 diagram  3. Design and disseminate advertisement for vacancy.
  p32 diagram  4. Decide on stages of selection and special weightage criteria, if any.
  p32 diagram  5. Review CVs and shortlist applications.
  p32 diagram  6. Proceed through all selection stages.
  p32 diagram  7. Reference check of selected candidates.


## 20.5

In [17]:
import re, json
from collections import Counter
import pandas as pd
from IPython.display import display
from src.paths import SECTIONS

secs = [json.loads(line) for line in SECTIONS.open(encoding="utf-8")]
print(f"loaded {len(secs)} sections from {SECTIONS.name}")

# ---------- 0. was 'top' saved? ----------
lines_all = [l for s in secs for l in s["lines"]]
print(f"\n0. lines with a 'top' value: {sum(l.get('top') is not None for l in lines_all)} / {len(lines_all)}")

# ---------- 1. section sizes (for chunk size) ----------
sizes = pd.DataFrame([{"id": s["section_id"], "kind": s["kind"], "label": s["label"][:55],
                       "pages": len(s["pdf_pages"]), "tables": len(s["tables"]), "chars": s["n_chars"]}
                      for s in secs])
print("\n1. chars per section:", sizes["chars"].describe().round(0).to_dict())
print("   sections by size:", pd.cut(sizes["chars"], [-1, 0, 500, 1000, 2000, 4000, 8000, 10**6],
                                     labels=["0", "≤500", "≤1k", "≤2k", "≤4k", "≤8k", ">8k"]).value_counts(sort=False).to_dict())
display(sizes.sort_values("chars", ascending=False).head(10))

# ---------- 2. lines ending with '-' and the line after them ----------
pairs = [(s["section_id"], a["text"][-20:] + " | " + b["text"][:25])
         for s in secs for a, b in zip(s["lines"], s["lines"][1:])
         if a["level"] == "body" and re.search(r"[a-z]-$", a["text"])]
print(f"\n2. lines ending with '-': {len(pairs)}")
for p in pairs:
    print("  ", p)

# ---------- 3. gaps between neighbouring body lines (for paragraph breaks) ----------
gaps = Counter()
for s in secs:
    for a, b in zip(s["lines"], s["lines"][1:]):
        if (a["level"] == b["level"] == "body" and a["pdf_page"] == b["pdf_page"]
                and a.get("top") is not None and b.get("top") is not None):
            gaps[round(b["top"] - a["top"])] += 1
print("\n3. gap between body lines → how often:", sorted(gaps.items())[:25])

loaded 106 sections from sections.jsonl

0. lines with a 'top' value: 3195 / 3203

1. chars per section: {'count': 106.0, 'mean': 2196.0, 'std': 4333.0, 'min': 29.0, '25%': 587.0, '50%': 1047.0, '75%': 2242.0, 'max': 40259.0}
   sections by size: {'0': 0, '≤500': 21, '≤1k': 29, '≤2k': 24, '≤4k': 17, '≤8k': 11, '>8k': 4}


,id,kind,label,pages,tables,chars
72,S073,annexure item,Sample ToRs/ Roles and Responsibilities,17,0,40259
78,S079,annexure item,Process of Performance Appraisal,8,11,14203
86,S087,annexure item,Process of Rationalization,2,2,10593
75,S076,annexure item,Contents of the Contract,4,1,10228
88,S089,annexure item,Exit Policy,6,8,5790
97,S098,annexure item,Appraisals and Rationalization of Human Resour...,3,1,5085
98,S099,annexure item,Person-Centred Care through Shared leadership-...,3,0,4904
73,S074,annexure item,Stages of Selection,2,0,4891
103,S104,annexure item,AMANAT - Nurse Mobile Mentoring Programme - Bihar,2,0,4848
71,S072,annexure item,Process of Designing Job Description/ToR,2,1,4646



2. lines ending with '-': 13
   ('S041', 'a simpler knowledge- | based training could take')
   ('S049', 'ons for performance- | based incentives. States ')
   ('S065', 'ided through a well- | functioning HRMIS.')
   ('S071', 'lications of the in- | service candidates would ')
   ('S073', 'y when there is non- | availability of General M')
   ('S079', 'ts according to pre- | established deadlines or ')
   ('S085', ' nodal officer / in- | charge of Human Resource ')
   ('S093', ' developed using in- | house capacity.')
   ('S097', 'mpleted during June- | July, 2016.')
   ('S098', 'the workforce. Self- | Declaration formats with ')
   ('S099', 'w to provide person- | centred care. Furthermore')
   ('S104', 'ure, and without in- | service training of avail')
   ('S105', 'lists (Anaesthetist- | Gynaecologist). During th')

3. gap between body lines → how often: [(13, 33), (14, 1665), (16, 1), (18, 4), (19, 632), (20, 13), (22, 7), (23, 261), (25, 2), (32, 1), (37, 1), (42, 1), (46, 25), (

## 20.T Tests

In [18]:
n_pass, report = -1, []                 # a crash below can never leave an old "all passed" behind
TESTS = []
def check(group, name, get_actual, expected, desc=None):
    try:
        actual = get_actual()
        ok = expected(actual) if callable(expected) else actual == expected
    except Exception as e:
        actual, ok = f"ERROR {type(e).__name__}: {e}", False
    TESTS.append({"group": group, "test": name, "result": "PASS" if ok else "FAIL",
                  "expected": desc if callable(expected) else expected, "actual": actual})

def raises(fn):
    try:
        fn(); return "no error"
    except Exception as e:
        return type(e).__name__

IS_PARENT = {s["section_id"]: k + 1 < len(STARTS) and STARTS[k + 1]["level"] > s["level"]
             for k, s in enumerate(STARTS)}
BY_NUM   = {s["number"]: s for s in SECTIONS_OUT if s["kind"] == "section"}
CS_PARTS = [s for s in SECTIONS_OUT if s["kind"] == "case study part"]
P2 = next(s for s in CS_PARTS if s["pdf_pages"][:1] == [148])     # Part 2 on PDF 148
P1 = CS_PARTS[CS_PARTS.index(P2) - 1]                              # the part right before it

G = "20 starts"
check(G, "105 TOC rows used (no letters, no Annexure II)", lambda: len(TOC_ROWS), 105)
check(G, "every TOC row found its heading", lambda: NOT_FOUND, [])
check(G, "106 sections", lambda: len(SECTIONS_OUT), 106)
check(G, "starts in page order",
      lambda: all((a["pdf_page"], a["line"]) < (b["pdf_page"], b["line"]) for a, b in zip(STARTS, STARTS[1:])), True)
check(G, "each section starts on its TOC page",
      lambda: [s["section_id"] for s in SECTIONS_OUT
               if s["kind"] != "glossary" and s["printed_pages"][:1] != [str(s["printed_start"])]], [])

G = "20 lines"
check(G, "no line before the first section", lambda: UNASSIGNED, [])
check(G, "every searchable line in exactly one section",
      lambda: sum(len(s["lines"]) for s in SECTIONS_OUT), sum(len(PAGE[n]["lines"]) for n in SEARCH))
check(G, "every table in exactly one section",
      lambda: sorted(t["id"] for s in SECTIONS_OUT for t in s["tables"]),
      sorted(t["id"] for n in SEARCH for t in PAGE[n]["tables"]))
check(G, "no empty section (except a parent followed by its child)",
      lambda: [s["section_id"] for s in SECTIONS_OUT
               if not IS_PARENT[s["section_id"]] and all(l["level"] in HEADING for l in s["lines"])], [])
check(G, "lines keep their 'top' position",
      lambda: sum(l.get("top") is not None for s in SECTIONS_OUT for l in s["lines"]) > 0, True)

G = "20 real cases"
check(G, "abbreviations = PDF 15-16",
      lambda: (SECTIONS_OUT[0]["title"], SECTIONS_OUT[0]["pdf_pages"]), ("List of Abbreviations", [15, 16]))
check(G, "PDF 32 holds end of 5.1, then 5.2, 5.3, 5.3.1",
      lambda: [s["number"] for s in SECTIONS_OUT if 32 in s["pdf_pages"]], ["5.1", "5.2", "5.3", "5.3.1"])
check(G, "5.3 holds the 8 manual diagram steps",
      lambda: sum(l["level"] == "diagram" for l in BY_NUM["5.3"]["lines"]), 8)
check(G, "path of 5.3.1", lambda: BY_NUM["5.3.1"]["path"],
      ["Chapter 5: Recruitment", "5.3 Process of Recruitment", "5.3.1 Identification and Validation of Vacancies"])
check(G, "PDF 148: Part 2 starts at its 'Part 2' line",
      lambda: P2["lines"][0]["text"][:6].lower(), "part 2")
check(G, "PDF 148: 'Advantages and Scalability' stays in the part before",
      lambda: any(l["text"].startswith("Advantages and Scalability") for l in P1["lines"]), True)
check(G, "no section ends with a bare 'Annexure' / 'III' line",
      lambda: [s["section_id"] for s in SECTIONS_OUT if s["lines"] and LABEL_PART.match(s["lines"][-1]["text"].strip())], [])
check(G, "every annexure section starts with its 'Annexure' line",
      lambda: [s["section_id"] for s in SECTIONS_OUT
               if s["kind"] == "annexure" and not s["lines"][0]["text"].lower().startswith("annexure")], [])

G = "20 unit"
SEC_53 = {"kind": "section", "number": "5.3", "title": "Process of Recruitment"}
check(G, "match: section by number", lambda: match_score(SEC_53, "5.3. Process of Recruitment"), 2)
check(G, "match: 5.3.1 is not 5.3", lambda: match_score(SEC_53, "5.3.1. Identification and Validation of Vacancies"), 0)
check(G, "match: 'Part 2' despite spacing",
      lambda: match_score({"kind": "case study part", "number": "2", "title": "Swasthya Sewa Dapoon"},
                          "Part 2: SwasthyaSewa Dapoon"), 2)
check(G, "match: chapter title, any case",
      lambda: match_score({"kind": "chapter", "number": "5", "title": "Recruitment"}, "RECRUITMENT"), 3)
check(G, "label pieces: 'Annexure', 'I', 'III', 'Chapter 5' yes · a title no",
      lambda: [bool(LABEL_PART.match(t)) for t in ["Annexure", "I", "III", "Chapter 5", "Suggestive Procedural Formats"]],
      [True, True, True, True, False])
check(G, "unknown kind fails loudly", lambda: raises(lambda: LABEL["poster"]), "KeyError")
check(G, "tuple order", lambda: (32, 5) > (32, 2) and (33, 0) > (32, 40), True)

report = pd.DataFrame(TESTS)
n_pass = int((report["result"] == "PASS").sum())
print(f"{'✅' if n_pass == len(report) else '❌'}  {n_pass}/{len(report)} tests passed")
failed = report[report["result"] == "FAIL"]
failed if len(failed) else "All tests passed — nothing to show."

✅  25/25 tests passed


'All tests passed — nothing to show.'

## 20.H Save, only when all tests pass

In [19]:
if n_pass == len(report):
    with SECTIONS.open("w", encoding="utf-8") as f:
        for s in SECTIONS_OUT:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    assert read_jsonl(SECTIONS) == SECTIONS_OUT, "reload differs from what was saved"
    print(f"saved {len(SECTIONS_OUT)} sections → {SECTIONS.name} (reload identical)")
else:
    print("NOT saved — fix the failing tests first")

saved 106 sections → sections.jsonl (reload identical)


In [20]:
secs = read_jsonl(SECTIONS)
for sid, part in [("S089", slice(-5, None)), ("S090", slice(0, 4))]:
    s = next(x for x in secs if x["section_id"] == sid)
    print(f"\n{sid} · {s['kind']} · {s['label'][:50]} · PDF {s['pdf_pages']}")
    for l in s["lines"][part]:
        print(f"   p{l['pdf_page']} {l['level']:<7} top={l.get('top')} | {l['text'][:60]}")


S089 · annexure item · Exit Policy · PDF [104, 105, 106, 107, 108]
   p107 table   top=244.6 | [table 107-1]
   p108 table   top=54.5 | [table 108-1]
   p108 body    top=294.8 | ----------------------------------------------TO BE COMPLETE
   p108 body    top=340.8 | Received On: Received By:
   p108 body    top=386.8 | Relieving Status: Relieving Letter Issued On:

S090 · annexure · Annexure III: Learnings from Other States · PDF [146]
   p146 L1      top=53.1 | Annexure
   p146 body    top=77.2 | III
   p146 L0      top=191.7 | Learnings from Other States
   p146 body    top=314.9 | India is a highly diverse country and the nature and spectru
